In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from tensorflow import keras
import os
import numpy as np

### Appliance energy consumption

Effectively same procedure used in the pioneering problem, bikes.

Adapted methods to now use past target values as inputs, autoregressive learning problem.

Observations come in 10-minute grid and will forecast the next `Appliances` value (Wh).

In [ ]:
dataset_name = 'Appliances'
target_unit = 'Wh / 10 minutes'
nonnegative_target = True
rolling_window = 7 * 24 * 6
frame = pd.read_csv('appliances/energydata_complete.csv', usecols=['date', 'Appliances'])
frame['date'] = pd.to_datetime(frame['date'])
time_col, target_col, frequency = 'date', 'Appliances', '10min'
series = pd.Series(frame[target_col].to_numpy(dtype=float),
                   index=pd.DatetimeIndex(frame[time_col]), name=target_col).sort_index()
groups = series.groupby(level=0)
series = groups.first().mask(groups.nunique(dropna=False).gt(1))
target = series.asfreq(frequency).replace([np.inf, -np.inf], np.nan)
test_start = int(0.8 * len(target))

development = target.iloc[:test_start]
target.head()

### ADF test on sampled long sequence to test for stationarity

Uses the longest contiguous observed segment for ACF and ADF so missing timestamps are not joined together

In [ ]:
segments = development.notna().ne(development.notna().shift()).cumsum()
observed = development.dropna()
longest_segment = segments.loc[observed.index].value_counts().idxmax()
diagnostic_series = observed.loc[segments.loc[observed.index].eq(longest_segment)]

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf

fig, ax = plt.subplots(figsize=(10, 4))
plot_acf(diagnostic_series, lags=200, ax=ax)
plt.show()

In [ ]:
from statsmodels.tsa.stattools import adfuller

def run_adf(series, label):
    result = adfuller(series.dropna(), autolag='AIC')
    print(f'{label}')
    print(f'  ADF statistic: {result[0]:.4f}')
    print(f'  p-value:       {result[1]:.4f}')
    print(f'  # lags used:   {result[2]}')
    print(f'  # obs used:    {result[3]}')
    for pct, crit in result[4].items():
        print(f'  critical value ({pct}): {crit:.4f}')
    truth = True if result[1] < 0.05 else False
    print(f'  Stationary at 5%? {truth}')
    print()
    return result

adf_result = run_adf(diagnostic_series, 'Raw development target')

### Rolling mean and standard deviation

Can be helpful to identify if ADF rejection is false, because seasonal trends obscure adf.

In [ ]:
rolling = development.rolling(rolling_window, min_periods=rolling_window)
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
axes[0].plot(development, linewidth=0.5)
axes[0].set(title=dataset_name, ylabel=target_unit)
axes[1].plot(rolling.mean())
axes[1].set(ylabel='Rolling mean')
axes[2].plot(rolling.std())
axes[2].set(ylabel='Rolling std.', xlabel='Time / sample')
fig.tight_layout()
plt.show()

### Features and target

One feature: past target values

In [ ]:
from sklearn.preprocessing import StandardScaler
from IPython.display import display

features = target.to_frame()
target_values = target.copy()
print('Input features:', features.columns.tolist())

### Five separate chronological blocks same as before, friend

In [ ]:
sequence_lengths = [6, 36, 144]  # dataset-scale lookbacks

blocks = []
for positions in np.array_split(np.arange(test_start), 5):
    start, end = int(positions[0]), int(positions[-1] + 1)
    train_end = start + int(0.8 * (end - start))
    blocks.append((start, train_end, end))

display(pd.DataFrame([
    {'fold': i + 1, 'block_start': target.index[start],
     'early_stop_start': target.index[start + int(0.85 * (train_end - start))],
     'validation_start': target.index[train_end], 'block_end': target.index[end - 1]}
    for i, (start, train_end, end) in enumerate(blocks)
]))

## Pre-processing

The bike block-preparation function, with raw levels as the target. 
Sepcifically:
Skip windows with a missing element/copied forward time
Scaling inside training segment
All lookbacks use the same eligible target positions since checks based on maxmimum sequence len

In [ ]:
def prepare_block(start, train_end, end, length):
    assert length in sequence_lengths
    fit_end = int(0.85 * (train_end - start))
    validation_start = train_end - start

    x = features.iloc[start:end].to_numpy()
    y = target_values.iloc[start:end].to_numpy()
    counts = target.iloc[start:end].to_numpy()

    longest = max(sequence_lengths)
    valid = np.array([
        t for t in range(longest, len(x))
        if np.isfinite(x[t-longest:t]).all() and np.isfinite(y[t])
    ], dtype=int)
    groups = {
        'train': valid[valid < fit_end],
        'stop': valid[(valid >= fit_end) & (valid < validation_start)],
        'validation': valid[valid >= validation_start]
    }
    assert all(len(ids) for ids in groups.values()), 'A split has no usable sequences.'

    x_scaler = StandardScaler().fit(x[:fit_end][np.isfinite(x[:fit_end]).all(axis=1)])
    y_scaler = StandardScaler().fit(y[groups['train'], None])
    x_scaled = x_scaler.transform(x)
    prepared = {}
    for name, ids in groups.items():
        prepared[name] = {
            'X': np.stack([x_scaled[t-length:t] for t in ids]).astype('float32'),
            'y': y_scaler.transform(y[ids, None]).astype('float32'),
            'last_value': counts[ids - 1],
            'actual': counts[ids],
            'dates': target.index[start + ids]
        }
    return prepared, x_scaler, y_scaler

### First block, 36-step sequences


In [ ]:
fold_data, x_scaler, y_scaler = prepare_block(*blocks[0], length=36)
X_train, y_train = fold_data['train']['X'], fold_data['train']['y']
X_stop, y_stop = fold_data['stop']['X'], fold_data['stop']['y']
X_val, y_val = fold_data['validation']['X'], fold_data['validation']['y']

for name, part in fold_data.items():
    print(name, 'X:', part['X'].shape, 'y:', part['y'].shape)

fold_sizes = []
for i, block in enumerate(blocks, 1):
    prepared, _, _ = prepare_block(*block, length=36)
    fold_sizes.append({'fold': i, **{name: len(part['y']) for name, part in prepared.items()}})
display(pd.DataFrame(fold_sizes))

## Train Elman, Jordan and multi-recurrent RNNs on blocked CV5

Use four configurations: `[(6, 32), (36, 32), (36, 64), (144, 32)]`. At 10-minute sampling, 6, 36 and 144 steps represent one hour, six hours and one day. The 64-unit version is tested at the central six-hour lookback. This gives 20 fits per architecture (four configurations across five blocks). All three architectures use exactly the same search within this dataset. The fixed 32-unit baseline limits complexity; one 64-unit candidate checks capacity without duplicating both widths at every lookback.


In [ ]:
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt

test_preview, _, _ = prepare_block(0, test_start, len(target), length=36)
print('Final test slots:', len(target) - test_start)
print('Eligible final test targets:', len(test_preview['validation']['y']))
del test_preview

### Model definitions

**Elman:** the previous hidden state feeds back into the hidden layer (`SimpleRNN`).

**Jordan:** the previous predicted output feeds back into the hidden layer. Here each output is a scaled target value; a short custom cell implements this feedback. It starts at zero for each input window and never receives true target values.

**Multi-recurrent (MRNN):** one hidden layer with both hidden-state feedback and predicted-output feedback, as in the supplied diagram:

\[
h_t = \tanh(W_x x_t + W_h h_{t-1} + W_y \hat y_{t-1} + b_h),
\qquad \hat y_t = W_o h_t + b_o.
\]

`MultiRecurrentCell` stores `[hidden, output]`. Both states start at zero for each input window; feedback uses the model's own scaled output, never a true target. The diagram's constant -1 inputs are represented by trainable Dense biases. The hidden and output feedback signals remain deterministic during training. Dropout is applied only to the final hidden representation after the recurrent window, immediately before the readout, so dropped values are never propagated. With a one-step window there is no earlier within-window state to feed back, so also compare the longer lookbacks.

All models end in one linear regression output. Equal hidden widths do not imply equal parameter counts; these are reported.


In [ ]:
class JordanCell(keras.layers.Layer):
    def __init__(self, units):
        super().__init__()
        self.state_size = 1
        self.output_size = units
        self.hidden = keras.layers.Dense(units, activation='tanh')
        self.readout = keras.layers.Dense(1)

    def build(self, input_shape):
        self.hidden.build((None, int(input_shape[-1]) + 1))
        self.readout.build((None, self.hidden.units))
        super().build(input_shape)

    def call(self, inputs, states, training=None):
        combined = tf.concat([inputs, states[0]], axis=-1)
        hidden = self.hidden(combined)
        output = self.readout(hidden)
        # Expose hidden for the final readout; feed back the undropped prediction.
        return hidden, [output]


class MultiRecurrentCell(keras.layers.Layer):
    def __init__(self, units):
        super().__init__()
        self.state_size = [units, 1]  # Previous hidden state and predicted output.
        self.output_size = units
        self.hidden = keras.layers.Dense(units, activation='tanh')
        self.readout = keras.layers.Dense(1)

    def build(self, input_shape):
        self.hidden.build((None, int(input_shape[-1]) + self.state_size[0] + 1))
        self.readout.build((None, self.state_size[0]))
        super().build(input_shape)

    def call(self, inputs, states, training=None):
        previous_hidden, previous_output = states
        combined = tf.concat([inputs, previous_hidden, previous_output], axis=-1)
        hidden = self.hidden(combined)
        output = self.readout(hidden)
        # Both feedback states are calculated before final-output dropout.
        return hidden, [hidden, output]


def build_rnn(name, length, units):
    inputs = keras.Input(shape=(length, features.shape[1]))
    if name == 'Jordan':
        cell = JordanCell(units)
        hidden = keras.layers.RNN(cell)(inputs)
    elif name == 'MultiRecurrent':
        cell = MultiRecurrentCell(units)
        hidden = keras.layers.RNN(cell)(inputs)
    elif name == 'Elman':
        cell = None
        hidden = keras.layers.SimpleRNN(units)(inputs)
    else:
        raise ValueError(name)
    # Drop only the final representation; no dropped value enters a recurrent state.
    hidden = keras.layers.Dropout(0.1)(hidden)
    output = (cell.readout(hidden) if cell is not None
              else keras.layers.Dense(1)(hidden))
    model = keras.Model(inputs, output, name=name)
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.005, clipnorm=2.0),
                  loss=keras.losses.Huber(3.0), metrics=['mae'])
    return model


### Fit one model and measure its errors

Same training loop and settings as bikes. Invert scaling directly because the target is a level, not a log difference.

In [ ]:
def predictions(model, part, scaler):
    prediction = scaler.inverse_transform(model.predict(part['X'], verbose=0)).ravel()
    return np.maximum(0, prediction) if nonnegative_target else prediction


def fit_rnn(name, data, scaler, length, units, max_epochs=60):
    keras.backend.clear_session()
    keras.utils.set_random_seed(11)
    model = build_rnn(name, length, units)
    stopping = keras.callbacks.EarlyStopping(
        monitor='val_mae', patience=5, restore_best_weights=True)
    history = model.fit(
        data['train']['X'], data['train']['y'],
        validation_data=(data['stop']['X'], data['stop']['y']),
        epochs=max_epochs, batch_size=64, shuffle=True, verbose=0,
        callbacks=[stopping, keras.callbacks.TerminateOnNaN()]
    ).history
    assert np.isfinite(history['val_mae']).all(), 'Training became unstable.'
    prediction = predictions(model, data['validation'], scaler)
    actual = data['validation']['actual']
    error = prediction - actual
    scores = {
        'MAE': np.mean(np.abs(error)),
        'RMSE': np.sqrt(np.mean(error**2)),
        'best_epoch': int(np.argmin(history['val_mae']) + 1),
        'epochs_run': len(history['mae']),
        'hit_epoch_limit': len(history['mae']) == max_epochs,
        'parameters': model.count_params()
    }
    for split in ['train', 'stop']:
        scores[split + '_MAE'] = np.mean(np.abs(
            predictions(model, data[split], scaler) - data[split]['actual']))

    history['mae_original_units'] = (np.array(history['mae']) * scaler.scale_[0]).tolist()
    history['val_mae_original_units'] = (np.array(history['val_mae']) * scaler.scale_[0]).tolist()
    return model, scores, history, prediction

In [ ]:
def run_cv(configurations, name):
    rows, histories = [], {}
    for length, units in configurations:
        for fold, block in enumerate(blocks, 1):
            data, _, scaler = prepare_block(*block, length=length)
           
            print(f'{name}: length={length}, units={units}, fold={fold}/5', flush=True)
            _, scores, history, _ = fit_rnn(name, data, scaler, length, units)
            rows.append({'model': name, 'length': length, 'units': units,
                            'fold': fold, 'n_validation': len(data['validation']['y']), **scores})
            histories[(name, length, units, fold)] = history
            print(f"  MAE={scores['MAE']:.2f}, best epoch={scores['best_epoch']}", flush=True)
    
    return pd.DataFrame(rows), histories

def persistence_cv(blocks):
    """Prediction is simply target before time t - a passive model baseline effectively"""
    rows = []
    for fold, block in enumerate(blocks, 1):
        data, _, _ = prepare_block(*block, length=sequence_lengths[0])
        actual = data['validation']['actual']
        last_value = data['validation']['last_value']
        error = last_value - actual
        rows.append({
            'fold': fold,
            'n_validation': len(actual),
            'persistence_MAE': np.mean(np.abs(error)),
            'persistence_RMSE': np.sqrt(np.mean(error**2)),
            'validation_mean': np.mean(actual),
        })
    return pd.DataFrame(rows)

def persistence_comparison(cv_results, persistence_results):

    keys = ['model', 'length', 'units']
    expected_folds = set(persistence_results['fold'])
    for _, group in cv_results.groupby(keys):
        if (set(group['fold']) != expected_folds or group['fold'].duplicated().any()
                or not np.isfinite(group['MAE']).all()):
            raise ValueError('Each configuration must have one finite MAE per baseline fold.')
    ranking = (cv_results.groupby(keys, as_index=False)
               .agg(mean_CV_MAE=('MAE', 'mean'))
               .sort_values(['model', 'mean_CV_MAE', 'length', 'units']))
    best = ranking.drop_duplicates('model')
    selected = cv_results.merge(best, on=keys, validate='many_to_one')
    merged = selected.merge(persistence_results, on='fold',
                            suffixes=('', '_baseline'), validate='many_to_one')
    if not merged['n_validation'].eq(merged['n_validation_baseline']).all():
        raise ValueError('Recompute model and persistence results on the same validation targets.')
    merged['relative_MAE'] = merged['MAE'] / merged['validation_mean']
    merged['MASE_persistence'] = merged['MAE'] / merged['persistence_MAE']  # < 1 beats persistence
    merged['skill_vs_persistence'] = 1 - merged['MASE_persistence']  # > 0 beats persistence
    return merged.sort_values(['model', 'fold']).reset_index(drop=True)

persistence_results = persistence_cv(blocks)
display(persistence_results)

### Elman RNN

In [ ]:
configurations = [(6, 32), (36, 32), (36, 64), (144, 32)]
cv_results1, cv_histories1 = run_cv(configurations, name='Elman')
configuration_means1 = (
    cv_results1.groupby(['length', 'units'])[['train_MAE', 'MAE']]
    .mean().reindex(pd.MultiIndex.from_tuples(configurations, names=['length', 'units']))
)
mae_train_mean = configuration_means1['train_MAE'].tolist()
mae_eval_mean = configuration_means1['MAE'].tolist()

In [ ]:
comparison1 = persistence_comparison(cv_results1, persistence_results)
display(comparison1[['model', 'length', 'units', 'mean_CV_MAE', 'fold',
                       'MAE', 'persistence_MAE', 'RMSE', 'persistence_RMSE',
                       'MASE_persistence', 'skill_vs_persistence', 'relative_MAE']])

In [ ]:
print(np.mean(comparison1["MASE_persistence"]))

### Jordan RNN

In [ ]:
configurations = [(6, 32), (36, 32), (36, 64), (144, 32)]
cv_results2, cv_histories2 = run_cv(configurations, name='Jordan')
configuration_means2 = (
    cv_results2.groupby(['length', 'units'])[['train_MAE', 'MAE']]
    .mean().reindex(pd.MultiIndex.from_tuples(configurations, names=['length', 'units']))
)
mae_train_mean2 = configuration_means2['train_MAE'].tolist()
mae_eval_mean2 = configuration_means2['MAE'].tolist()

In [ ]:
comparison2 = persistence_comparison(cv_results2, persistence_results)
display(comparison2[['model', 'length', 'units', 'mean_CV_MAE', 'fold',
                       'MAE', 'persistence_MAE', 'RMSE', 'persistence_RMSE',
                       'MASE_persistence', 'skill_vs_persistence', 'relative_MAE']])

In [ ]:
print(np.mean(comparison2["MASE_persistence"]))

### MultiRecurrent RNN

In [ ]:
configurations = [(6, 32), (36, 32), (36, 64), (144, 32)]

cv_results3, cv_histories3 = run_cv(configurations, name='MultiRecurrent')
configuration_means3 = (
    cv_results3.groupby(['length', 'units'])[['train_MAE', 'MAE']]
    .mean().reindex(pd.MultiIndex.from_tuples(configurations, names=['length', 'units']))
)
mae_train_mean3 = configuration_means3['train_MAE'].tolist()
mae_eval_mean3 = configuration_means3['MAE'].tolist()

In [ ]:
comparison3 = persistence_comparison(cv_results3, persistence_results)
display(comparison3[['model', 'length', 'units', 'mean_CV_MAE', 'fold',
                       'MAE', 'persistence_MAE', 'RMSE', 'persistence_RMSE',
                       'MASE_persistence', 'skill_vs_persistence', 'relative_MAE']])

In [ ]:
print(np.mean(comparison3["MASE_persistence"]))

### Final chronological holdout


In [ ]:
cv_results_all = pd.concat([cv_results1, cv_results2, cv_results3], ignore_index=True)


best_configs = persistence_comparison(cv_results_all, persistence_results)[
    ['model', 'length', 'units']].drop_duplicates()

final_results = []
for _, row in best_configs.iterrows():
    name, length, units = row['model'], int(row['length']), int(row['units'])
    final_data, _, y_scaler = prepare_block(0, test_start, len(target), length=length)
    model, scores, history, prediction = fit_rnn(name, final_data, y_scaler, length, units)
    final_results.append({'model': name, 'length': length, 'units': units, **scores})

final_results = pd.DataFrame(final_results)
display(final_results)